In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom without
# reimporting mid-notebook.
#
# The PROCESSING is written out step by step in the cells below rather than
# imported, so every numeric step can be read and edited here: the epoching, the
# FastICA over channels, the sign/order convention and the plotting.
import os
import sys
import warnings
import zipfile
from pathlib import Path  # noqa: F401

import matplotlib.pyplot as plt
import mne
import numpy as np
import numpy.lib.format as npformat
import pandas as pd
import seaborn as sns
from scipy.stats import zscore
from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, display  # noqa: E402

# Figure renderers (shared with the CLI script) + the stage-00 contrast measure,
# imported on purpose: reusing it is what makes the numbers comparable with
# notebooks/00-preprocessing/assr_wavelet_pca_analysis.ipynb.
from scripts.run_assr_wavelet_pca import driven_contrast  # noqa: E402
from scripts.run_separate_assr_wavelet_ica_channel import (  # noqa: E402
    plot_participant_components,
    plot_specificity_overview,
)

# Imported ONLY for the optional consistency check in Step 5 — never used to
# compute anything this notebook reports.
from scripts.run_separate_assr_wavelet_ica_channel import (  # noqa: E402
    assr_specificity as _script_assr_specificity,
)
from scripts.run_separate_assr_wavelet_ica_channel import (  # noqa: E402
    frequency_contrast_profile as _script_contrast_profile,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    CoordinateSystems,
    ExperimentNames,
    MusicTypeVariants,
    PreprocessedDataVariants,
)
from src.preprocessing.pipeline import DatasetHandler  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# Separate (Per-Participant) Channel ICA on Stimulus-Locked ASSR Wavelet Power

## Scope

Run the channel decomposition **independently for every participant** and look, per
participant, at what each independent component is: its **stimulus-locked
time-frequency map** and its **scalp topography**. The motivating question is
whether the typical ASSR pattern is carried by one component or split across
several — this notebook produces the evidence to judge that by eye; it does not
score or count anything.

The mixing dimension is **channels** and the sample axis is the joint **frequency ×
time** plane — the same geometry as the channel-IVA notebook in `05-*`:

```
Per subject:  (n_channels, n_freqs, win)   — stimulus-locked trial average
Reshape:      (n_freqs × win,  n_channels)
              ─── samples ───  ── mixing ──
FastICA over channels -> N_ICA components
    (FastICA's own SVD whitening keeps the leading N_ICA channel directions,
     so no separate PCA step is needed — see Step 3)
```

Each IC therefore gives a spectro-temporal score map `(n_freqs, win)` — the
stimulus-locked TF map — plus one channel topography.

## Why not the existing analyses

| Notebook | Question it answers | Why it cannot answer this one |
|---|---|---|
| `05-*/wavelet_iva_channel` | which spectro-temporal source is **shared across** subjects | IVA forces one component index to mean the same thing in every subject, so a subject whose 40 Hz response is split over two modes has that absorbed into the alignment |
| `00-*/assr_wavelet_pca_analysis` | leading **channel-variance** modes per subject | PCA forces components orthogonal, so a 40 Hz steady state sharing a topography with a broad onset response is *mixed* across PCs by construction |

ICA drops orthogonality and separates by statistical independence instead, so a
split across two ICs is evidence about the data rather than an artefact of the
basis.

## The workflow, step by step

Each step is one markdown cell explaining it and one code cell doing it.
Everything numeric is written out here so you can change it in place:

| Step | Cell does | Change it to… |
|---|---|---|
| **1** Epoching | average fixed windows around each `fam+` onset | a different baseline, or keep single trials |
| **2** Load | stream the cache, z-score vs recording, trial-average | drop the z-scoring, pick other subjects |
| **3** **FastICA** | unmix the channel axis; forward patterns; `ic_variance` | **`N_ICA`, the algorithm, `fun` — or the whole decomposition** |
| **4** Sign & order | display-only flip, order by `ic_variance` | a different convention |
| **5** Montage | electrode positions remapped onto the cache's channel order | — |
| **6** Figures | per-IC TF map above its topography, per subject | layout, colour scales, markers |

Nothing is imported to do the processing — only `numpy`, `sklearn` and `mne`.

## After the final cell

| Variable | Shape | Description |
|----------|-------|-------------|
| `trial_avg` | `(S, C, F, win)` | Stimulus-locked trial averages, PID-ordered |
| `tf_maps` | `(S, K, F, win)` | IC score maps, sign-corrected and variance-ordered |
| `patterns` | `(S, K, C)` | Forward (mixing) channel topographies |
| `ic_variance` | `(S, K)` | Share of channel variance each IC accounts for |
| `retained` | `(S,)` | Channel variance the whitening truncation kept |

**Whole cohort**: this notebook reads a subject *subset* so it stays interactive.
`scripts/run_separate_assr_wavelet_ica_channel.py` streams the cache once and
decomposes every participant — but note it still uses the explicit PCA → FastICA
path and carries the ASSR scoring, so it is not a drop-in match for this notebook
any more.

## Configuration

In [ ]:
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO   # default per project convention
MUSIC_TYPE = MusicTypeVariants.ASSR

# ── Subjects to decompose (all channels are used) ────────────────────────────
# Subject indices into the concatenated array (CONCATENATED_PERSON_INDEX). Keep
# these LOW — the lazy reader scans from the start of the deflate stream, so
# [0, 1, 2] only decompresses the first 3 subject blocks of the ~50 GB cache.
SUBJECT_INDICES = [0, 1, 2]

# ── Decomposition size ───────────────────────────────────────────────────────
# N_ICA is the ONLY dimensionality knob: FastICA whitens the channel matrix by
# SVD and keeps its leading N_ICA directions, so it performs the channel
# reduction itself and no separate PCA step is needed (see Step 3).
#
# It is the dimensionality the independence assumption has to work in. Too high
# and the ICA splits one response across several noisy components; too low and it
# cannot separate the steady state from an overlapping onset response. On this
# dataset FastICA converges at N_ICA <= 6 and does NOT at 10 — a non-converged
# unmixing is an arbitrary point on the solver's path, so prefer a value that
# converges (Step 3 reports it per subject).
N_ICA = 10
RANDOM_STATE = 42     # FastICA's initial unmixing is random; this pins it
ICA_MAX_ITER = 2000
ICA_TOL = 1e-4
ICA_ALGORITHM = "parallel"   # or "deflation" — see Step 4
ICA_FUN = "logcosh"          # contrast function: "logcosh", "exp" or "cube"

# Extra FastICA fits from different random initialisations, used ONLY to put a
# min-max range on the ASSR component count. FastICA's fixed point is not unique
# when the trailing retained PCA directions are near degenerate — the normal
# situation here, ten components out of ~200 channels — and on real ASSR data the
# solver routinely does not meet its tolerance, so the count is worth reporting
# with error bars. Restart 0 repeats the primary fit (same seed) as a check.
N_RESTARTS = 5

# ── Stimulus-locked epoch window ─────────────────────────────────────────────
# Taken from the paradigm definition (src.definitions.constants.AssrEpoch) so
# every onset-locked ASSR analysis cuts the same window:
#   PRE_PAD_S  = 0.1 s baseline before onset
#   POST_PAD_S = 1.0 s after onset = 0.5 s stimulus + 0.5 s post-stimulus
# The post length is capped by the shortest inter-onset gap below, so no epoch
# can reach a neighbouring stimulus.
PRE_PAD_S = AssrEpoch.PRE_ONSET_S
POST_PAD_S = AssrEpoch.POST_ONSET_S
SFREQ = 250.0              # sampling rate of the concatenated / wavelet data

# ── ASSR criterion ───────────────────────────────────────────────────────────
ASSR_FREQ = 40.0           # expected steady-state frequency (Hz)
ASSR_HALFWIDTH_HZ = 2.0    # half-width of the band averaged as the response
# Frequencies within this distance of ASSR_FREQ are excluded from the out-of-band
# null, so wavelet band leakage does not inflate it. Must be >= the half-width.
ASSR_GUARD_HZ = 4.0
# Specificity z above which an IC counts as carrying the ASSR. Only affects the
# counts and the flags — the z-scores are tabulated unthresholded, so trying a
# different cut-off needs no re-read of the cache.
ASSR_Z_THRESHOLD = 3.0

# Z-score each channel's per-frequency power against the WHOLE recording before
# epoching: removes the 1/f tilt so the channel decomposition is not dominated by
# absolute low-frequency power. Set False to decompose raw power.
ZSCORE_VS_RECORDING = True

# ── Wavelet cache descriptor (matches the stored filename) ───────────────────
WAVELET_FREQ_SIG = "1.000_50.000_50"   # freqs[0]_freqs[-1]_n_freqs
N_WAVELET_FREQS = 50

# ── Plot saving ──────────────────────────────────────────────────────────────
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / EXPERIMENT.value
    / "separate_assr_wavelet_ica_channel"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

n_ica = N_ICA
assert ASSR_GUARD_HZ >= ASSR_HALFWIDTH_HZ, "The guard must not overlap the band."

print(f"Group          : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Subjects       : {SUBJECT_INDICES}")
print(f"Decomposition  : {n_ica} ICs over {ICA_ALGORITHM}/{ICA_FUN} "
      f"(FastICA whitens the channels itself; no separate PCA), "
      f"seed {RANDOM_STATE}")
print(f"Pre-onset pad  : {PRE_PAD_S} s")
print(f"Post-onset span: {POST_PAD_S} s "
      f"({AssrEpoch.STIMULUS_DURATION_S} s stimulus + "
      f"{AssrEpoch.POST_STIMULUS_S} s post-stimulus)")
print(f"ASSR criterion : {ASSR_FREQ} +/- {ASSR_HALFWIDTH_HZ} Hz vs out-of-band "
      f"beyond {ASSR_GUARD_HZ} Hz, z >= {ASSR_Z_THRESHOLD}")
print(f"Z-score vs rec : {ZSCORE_VS_RECORDING}")
print(f"Plots -> {PLOTS_DIR}")

## Load Concatenated Onsets, Metadata & Channel Names

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"

concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
meta_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / "wavelets"
    / "broadband"
    / f"{safe_label}__wavelet_power__{WAVELET_FREQ_SIG}__freqdim1.npz"
)
for p in (onsets_path, meta_path, wavelet_path):
    assert p.exists(), f"Missing expected file: {p}"

onsets = np.load(onsets_path)                        # (n_onsets,) shared sample idx
meta = pd.read_csv(meta_path, index_col=0)
gaps = np.diff(onsets)
print(f"Stimulus onsets : {onsets.shape}  range [{onsets.min()}, {onsets.max()}]")
print(f"Inter-onset gap : min {int(gaps.min())}, median {int(np.median(gaps))}, "
      f"max {int(gaps.max())} samples")

# Channel names are stored in the wavelet feature_names (layout: channel*N_FREQS).
with zipfile.ZipFile(wavelet_path) as _z:
    with _z.open("feature_names.npy") as _f:
        feature_names = npformat.read_array(_f, allow_pickle=True)
channel_names = [str(feature_names[i * N_WAVELET_FREQS]).split("@")[0]
                 for i in range(len(feature_names) // N_WAVELET_FREQS)]
n_channels = len(channel_names)
print(f"Channels parsed : {n_channels} (first={channel_names[0]}, "
      f"last={channel_names[-1]})")
meta[["SingleDataMetadata.PARTICIPANT_ID", "SingleDataMetadata.CONDITION",
      "SingleDataMetadata.CONCATENATED_PERSON_INDEX"]].head()

## Subset Selection & Epoch Window

The epoch comes from the **paradigm**, not from the observed jitter: `PRE_PAD_S`
of baseline before onset, then `POST_PAD_S` after it — the 0.5 s stimulus plus
0.5 s post-stimulus, so a response outlasting the stimulus stays visible. The
post-onset length is then **capped** by the shortest inter-onset gap, so no epoch
can reach a neighbouring stimulus.

`stim_mask` marks the driven interval and `base_mask` the pre-onset baseline. The
split matters: a ~1.1 s epoch around a 0.5 s stimulus is half silence, so a
steady-state estimate averaged over the whole post-onset window is diluted by
construction.

`band_mask` and `null_mask` are the frequency masks the ASSR criterion in Step 5
uses; they are defined here so every later cell reads them rather than
re-deriving them.

Subjects are sorted by participant ID before anything else, so every array axis
and every panel below is in PID order rather than in concatenated-index order.

In [ ]:
# Epoch window in samples: paradigm span, capped by the shortest inter-onset gap.
PRE = AssrEpoch.pre_onset_samples(SFREQ)
POST = AssrEpoch.post_onset_samples(SFREQ, min_gap=int(gaps.min()))
win = PRE + POST
epoch_times = np.arange(-PRE, POST) / SFREQ         # (win,) seconds, t=0 at onset
stim_mask = AssrEpoch.stimulus_mask(epoch_times)    # driven interval
base_mask = epoch_times < 0.0                       # pre-onset baseline

# Map subject indices -> participant labels via the concatenated metadata.
pidx_col = "SingleDataMetadata.CONCATENATED_PERSON_INDEX"
pid_col = "SingleDataMetadata.PARTICIPANT_ID"
idx_to_pid = dict(zip(meta[pidx_col], meta[pid_col].astype(str).str.zfill(3)))

# Sort by participant ID so every downstream array and panel is PID-ordered.
SUBJECT_INDICES = sorted(
    SUBJECT_INDICES, key=lambda si: int(idx_to_pid.get(si, "9999"))
)
subject_labels = [f"PSI{idx_to_pid.get(si, '???')}" for si in SUBJECT_INDICES]
n_subj = len(SUBJECT_INDICES)

wavelet_freqs = np.arange(1.0, N_WAVELET_FREQS + 1.0)   # 1..50 Hz (as cached)
# Frequency masks for the ASSR criterion (Step 5).
band_mask = np.abs(wavelet_freqs - ASSR_FREQ) <= ASSR_HALFWIDTH_HZ
null_mask = np.abs(wavelet_freqs - ASSR_FREQ) > ASSR_GUARD_HZ

print(f"Subjects (PID order): {list(zip(SUBJECT_INDICES, subject_labels))}")
print(f"Epoch window : {win} samples ({PRE} pre, {POST} post) = "
      f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s")
print(f"Driven / base: {int(stim_mask.sum())} / {int(base_mask.sum())} samples")
print(f"ASSR band    : {wavelet_freqs[band_mask].min():.0f}-"
      f"{wavelet_freqs[band_mask].max():.0f} Hz ({int(band_mask.sum())} bins) vs "
      f"{int(null_mask.sum())} out-of-band bins")
print(f"Samples per decomposition: {N_WAVELET_FREQS * win} "
      f"(freq x time bins), mixing over {n_channels} channels")
if win > int(gaps.min()):
    print(f"  WARNING: epoch ({win}) exceeds the shortest gap "
          f"({int(gaps.min())}) — the baseline reaches into the previous stimulus.")

## Step 1 — Epoching & the lazy reader

**`epoch_average`** is the whole epoching rule: average a fixed window around
every onset, skipping windows that fall off the edge of the recording. Change it
here to use a different baseline, a median instead of a mean, or to return the
individual epochs.

**Why the trial average and not single trials**: the target is the stimulus-locked
(evoked) pattern, and averaging first is what removes the induced,
non-phase-locked background from the covariance the ICA sees. Fitting on
concatenated single trials would decompose an evoked+induced mixture and answer a
different question — a legitimate one, but a different one.

**`load_wavelet_trial_averaged`** exists because `savez_compressed` stores
`data.npy` as a single deflate stream, so the cache cannot be sliced randomly: it
has to be decompressed sequentially. The reader streams one
`(n_freqs, n_times)` channel block at a time and immediately reduces it to its
small `(n_freqs, win)` trial average, so the ~50 GB file never lands in RAM (peak
= one block). All channels are kept — the channel decomposition needs the full
electrode set.

The CLI script's reader differs in one respect only: it decomposes *every* subject
in one pass, while this one collects an arbitrary **subset**.

In [ ]:
def epoch_average(arr, onsets, pre, post):
    """Average fixed windows around each onset along the LAST axis.

    Args:
        arr: array whose last axis is time, e.g. ``(n_freqs, n_times)``.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).

    Returns:
        ``(averaged, n_used)`` where the time axis is replaced by the
        ``pre + post`` window, averaged over every onset whose window fits inside
        the recording (edge windows are skipped).
    """
    n_time = arr.shape[-1]
    acc = None
    n_used = 0
    for o in onsets:
        s, e = o - pre, o + post
        if s < 0 or e > n_time:
            continue
        seg = arr[..., s:e]
        acc = seg.astype(np.float64) if acc is None else acc + seg
        n_used += 1
    if n_used == 0:
        raise ValueError("No onset window fits inside the recording.")
    return acc / n_used, n_used


def load_wavelet_trial_averaged(npz_path, subject_indices, n_channels, n_freqs,
                               onsets, pre, post, *, zscore_time=True):
    """Lazily read a huge wavelet npz and trial-average EVERY channel per subject.

    Args:
        npz_path: path to the ``*__wavelet_power__*__freqdim1.npz`` cache.
        subject_indices: concatenated-array subject indices to collect.
        n_channels, n_freqs: cache dimensions.
        onsets: stimulus onset sample indices.
        pre, post: samples kept before / after each onset (window = pre + post).
        zscore_time: z-score each channel's per-frequency power against the whole
            recording before epoching (removes the 1/f tilt).

    Returns:
        ``(data, n_times, n_used)`` where ``data`` has shape
        ``(len(subject_indices), n_channels, n_freqs, pre + post)`` in the order
        *subject_indices* was given, and ``n_used`` maps subject idx -> number of
        stimuli averaged.
    """
    subj_set = set(subject_indices)
    max_subj = max(subject_indices)
    window = pre + post
    collected = {si: np.empty((n_channels, n_freqs, window)) for si in subject_indices}
    n_used = {}
    with zipfile.ZipFile(npz_path) as z:
        with z.open("data.npy") as f:
            ver = npformat.read_magic(f)
            if ver == (1, 0):
                shape, _, dtype = npformat.read_array_header_1_0(f)
            else:
                shape, _, dtype = npformat.read_array_header_2_0(f)
            n_subj_total, n_feat_flat, n_times = shape
            assert n_feat_flat == n_channels * n_freqs, (
                f"feature axis {n_feat_flat} != n_channels*n_freqs "
                f"{n_channels * n_freqs}"
            )
            assert max_subj < n_subj_total, (
                f"subject index {max_subj} is outside the cache "
                f"({n_subj_total} subjects)."
            )
            block_bytes = n_freqs * n_times * dtype.itemsize  # one channel block
            for s in range(max_subj + 1):
                for c in range(n_channels):
                    buf = f.read(block_bytes)
                    if s not in subj_set:
                        continue
                    block = np.frombuffer(
                        buf, dtype=dtype, count=n_freqs * n_times
                    ).reshape(n_freqs, n_times)
                    if zscore_time:
                        block = zscore(block, axis=1)
                    ev, used = epoch_average(block, onsets, pre, post)  # (f, win)
                    collected[s][c] = ev
                    n_used[s] = used
                if s in subj_set:
                    print(f"  scanned subject block {s} ...")
    data = np.stack([collected[si] for si in subject_indices])
    return data, n_times, n_used


print("Step 1 helpers defined.")

## Step 2 — Load & Trial-Average the Wavelet Power

This is the slow cell — everything after it is fast, so iterate on the
decomposition (Step 4) and the criterion (Step 5) without re-running it. Progress
is printed per subject block; the result is a small
`(n_subj, n_channels, n_freqs, win)` array.

In [ ]:
trial_avg, n_times_wav, n_used_w = load_wavelet_trial_averaged(
    wavelet_path,
    subject_indices=SUBJECT_INDICES,
    n_channels=n_channels,
    n_freqs=N_WAVELET_FREQS,
    onsets=onsets,
    pre=PRE,
    post=POST,
    zscore_time=ZSCORE_VS_RECORDING,
)
print(f"Trial-averaged wavelet: {trial_avg.shape}  (n_times_full={n_times_wav})")
for si, label in zip(SUBJECT_INDICES, subject_labels):
    print(f"  {label}: averaged {n_used_w[si]} stimuli")

## Step 3 — FastICA over Channels  ⟵ *the decomposition; change it here*

Each subject's trial average is reshaped so every `(freq, time)` bin is a
**sample** and every channel a **mixing variable** — matrix `X` of shape
`(n_freqs*win, n_channels)` — and FastICA unmixes the channel axis directly.

**There is no separate PCA step, and none is needed.** `FastICA(n_components=K)`
whitens `X` by SVD and keeps its leading `K` directions, which is the same
subspace a `PCA(K)` would have produced — so an explicit PCA was reducing to a
basis FastICA picks anyway. Verified on this data: the two paths span an identical
subspace (‖ΔP‖ = 1e-14), retain the same variance to 14 significant figures, and
return the same components wherever FastICA converges (|r| = 1.000 at `N_ICA=3`,
0.999 at 6). The reduction did not disappear — it moved inside `FastICA`, where
`N_ICA` controls it.

Things worth trying here:

- `ICA_ALGORITHM = "deflation"` extracts components one at a time and often
  converges where the symmetric `"parallel"` update oscillates.
- `ICA_FUN` picks the contrast function (`"logcosh"`, `"exp"`, `"cube"`);
  `"cube"` (kurtosis) favours spiky sources, `"exp"` heavier tails.
- Replacing `FastICA` entirely — Infomax via `mne.preprocessing.ICA`, or Picard —
  needs nothing else changed as long as the function still returns
  `(tf_maps, patterns, ic_variance, retained, converged)` with the same shapes.

Four outputs deserve attention:

**`tf_maps`** — each IC's score map, reshaped back to `(n_freqs, win)`. FastICA's
`unit-variance` whitening fixes every source to unit variance, so a subject's maps
share a scale and a common colour limit is fair; the amplitude lives in the
pattern instead.

**`patterns`** — the **forward (mixing)** channel pattern, now simply
`ica.mixing_.T`, since without the PCA basis there is nothing to project back
through. This is what belongs on a topomap. The unmixing rows (`components_`) are
spatial *filters* — a different object, and plotting them instead is the classic
filter-vs-pattern error (Haufe et al., 2014, NeuroImage 87:96-110). MNE follows
the same convention: `ica.get_components()` returns the mixing matrix, not
`unmixing_matrix_`. `mixing_` is `pinv(components_)`, and because `components_`
carries the whitening folded in, that pseudo-inverse undoes the whitening too.

**`retained`** — the fraction of channel variance the whitening truncation kept,
recovered from the back-projection residual. It is the hard ceiling on everything
the ICs can carry: whatever is orthogonal to the retained subspace is gone. This
replaces the PCA's `explained_variance_ratio_.sum()` and equals it exactly.

**`ic_variance`** — the energy of each IC's rank-one back-projection over the
total channel-space sum of squares. `sources @ patterns` reconstructs the retained
part of the centred channel matrix, so these are the energies of its rank-one
terms; they sum to `retained` insofar as the sources are uncorrelated. They matter
because a unit-variance source says nothing about how much of the data the
component actually accounts for.

Non-convergence is captured rather than printed and lost. On this dataset it
happens at `N_ICA=10` and not at 6, and raising `ICA_MAX_ITER` makes it *worse*,
not better — the solver wanders rather than closing in. Lower `N_ICA` instead.

In [ ]:
def fit_channel_ica(matrix, n_ica, n_freqs, window, seed):
    """FastICA over the channel axis of one subject's trial average.

    Args:
        matrix: ``(n_freqs*window, n_channels)`` samples-by-channels matrix.
        n_ica: independent components to extract. Also sets the whitening
            truncation, since FastICA reduces the channel axis itself.
        n_freqs, window: shape to fold the sources back into a TF map.
        seed: random_state for FastICA's initial unmixing.

    Returns:
        ``(tf_maps, patterns, ic_variance, retained, converged)`` with shapes
        ``(n_ica, n_freqs, window)``, ``(n_ica, n_channels)``, ``(n_ica,)``,
        float, bool.
    """
    ica = FastICA(
        n_components=n_ica,
        algorithm=ICA_ALGORITHM,
        fun=ICA_FUN,
        whiten="unit-variance",
        random_state=seed,
        max_iter=ICA_MAX_ITER,
        tol=ICA_TOL,
    )
    # Capture ConvergenceWarning instead of letting it print once and vanish: a
    # non-converged unmixing is an arbitrary point on the solver's path.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        sources = ica.fit_transform(matrix)          # (n_freqs*window, n_ica)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)

    # Forward model, directly in channel space: mixing_ = pinv(components_), and
    # components_ carries the whitening, so this pseudo-inverse undoes it too.
    patterns = ica.mixing_.T                         # (n_ica, n_channels)
    tf_maps = sources.T.reshape(n_ica, n_freqs, window)

    centred = matrix - matrix.mean(axis=0)
    total_ss = float((centred ** 2).sum())
    if total_ss <= 0.0:
        return tf_maps, patterns, np.full(n_ica, np.nan), np.nan, converged
    # What the whitening truncation kept — the PCA's explained_variance_ratio_
    # sum, recovered from the residual now that there is no PCA object.
    retained = 1.0 - float(((centred - sources @ patterns) ** 2).sum()) / total_ss
    ic_variance = (sources ** 2).sum(axis=0) * (patterns ** 2).sum(axis=1) / total_ss
    return tf_maps, patterns, ic_variance, retained, converged


tf_maps = np.empty((n_subj, n_ica, N_WAVELET_FREQS, win))
patterns = np.empty((n_subj, n_ica, n_channels))
ic_variance = np.empty((n_subj, n_ica))
retained = np.empty(n_subj)
non_converged = []

for k, label in enumerate(subject_labels):
    # Samples = (freq, time) bins, mixing variables = channels.
    X = trial_avg[k].reshape(n_channels, N_WAVELET_FREQS * win).T
    (
        tf_maps[k], patterns[k], ic_variance[k], retained[k], converged,
    ) = fit_channel_ica(X, n_ica, N_WAVELET_FREQS, win, RANDOM_STATE)
    if not converged:
        non_converged.append(label)
    print(f"  {label}: X={X.shape} -> {n_ica} ICs, retained "
          f"{retained[k] * 100:.1f}% of channel variance, top ic_variance "
          f"{ic_variance[k].max() * 100:.1f}%, sum {ic_variance[k].sum() * 100:.1f}%"
          f"{'' if converged else '  <-- FastICA DID NOT CONVERGE'}")

if non_converged:
    print(f"\nWARNING: FastICA did not converge for {non_converged} within "
          f"{ICA_MAX_ITER} iterations at tol={ICA_TOL:g}, so their unmixing is "
          f"wherever the solver stopped rather than a fixed point. On this "
          f"dataset N_ICA <= 6 converges and 10 does not, and raising "
          f"ICA_MAX_ITER makes the result drift further rather than settle — "
          f"lower N_ICA, or try ICA_ALGORITHM='deflation'.")

## Step 4 — Display Sign & Order

FastICA fixes neither the sign nor the order of its components, so both have to be
pinned before plotting — otherwise the same data replotted with a different seed
looks like a different result. Neither choice is a claim about the data:

- **Sign.** `(map, pattern)` and `(−map, −pattern)` are the *same* component, so
  the sign is free. Convention here: flip each IC so the **largest absolute
  excursion of its TF map is positive**, i.e. each panel's dominant event reads as
  a power *increase*. This is deliberately hypothesis-free — it says nothing about
  40 Hz — so it stays valid whatever you go on to measure. An IC whose two largest
  excursions are near-equal in magnitude has an effectively arbitrary sign; nothing
  downstream should depend on it.
- **Order.** FastICA's output order is meaningless, so ICs are sorted by
  `ic_variance` descending: `IC1` accounts for the largest share of that subject's
  channel-space energy. That is a statement about *size*, not about relevance — a
  40 Hz steady state can easily sit below a broad onset response, so read all of
  them rather than the first few.

Both operations leave the back-projection `Σ_k source_k ⊗ pattern_k` untouched,
which is what the assertion checks: flipping and reordering rearrange the
bookkeeping, not the decomposition.

⚠️ `IC1` is a **subject-local** label. With independent per-subject decompositions
there is no reason subject A's `IC1` and subject B's `IC1` are the same component;
that identity is what the joint IVA variant in `05-*` provides and this analysis
deliberately does not.

In [ ]:
def back_projection(tf, pat):
    """Sum over components of source ⊗ pattern — invariant to sign and order.

    Args:
        tf: ``(S, K, F, win)`` component score maps.
        pat: ``(S, K, C)`` forward channel patterns.

    Returns:
        ``(S, F*win, C)`` reconstruction of the retained channel data.
    """
    return np.einsum("skn,skc->snc", tf.reshape(tf.shape[0], tf.shape[1], -1), pat)


_recon_before = back_projection(tf_maps, patterns)

# ── Sign: make each IC's largest absolute TF excursion positive ──────────────
_flat = tf_maps.reshape(n_subj, n_ica, -1)
_peak = np.take_along_axis(
    _flat, np.abs(_flat).argmax(axis=2)[..., np.newaxis], axis=2
)[..., 0]
signs = np.where(_peak < 0.0, -1.0, 1.0)                     # (S, K)
tf_maps = tf_maps * signs[..., np.newaxis, np.newaxis]
patterns = patterns * signs[..., np.newaxis]

# ── Order: largest share of channel-space energy first ──────────────────────
order = np.argsort(-ic_variance, axis=1, kind="stable")      # (S, K)
rows = np.arange(n_subj)[:, np.newaxis]
tf_maps = tf_maps[rows, order]
patterns = patterns[rows, order]
ic_variance = ic_variance[rows, order]

# Neither the flip nor the reordering may change what the components add up to.
np.testing.assert_allclose(
    back_projection(tf_maps, patterns), _recon_before, atol=1e-9,
    err_msg="Re-orienting or reordering changed the back-projection — the sign "
            "was not applied to the map and the pattern together.",
)
print(f"Flipped {int((signs < 0).sum())}/{signs.size} (subject, IC) pairs; "
      f"reordered by ic_variance. Back-projection unchanged.")
for label, v in zip(subject_labels, ic_variance):
    print(f"  {label}: ic_variance % = "
          + " ".join(f"{x * 100:5.1f}" for x in v))

## Step 5 — Topomap Electrode Positions

The patterns are indexed by the wavelet cache's channel order, which is not the
order any `Info` object uses, so a remap is needed before anything is drawn.
Positions come from one `RAW_CROPPED` recording's `Info`, read with
`preload=False` so only the header is touched and no sample data is loaded.

`info_order` indexes a pattern vector (cache order) into the montage's order, so
`pattern[info_order]` is what `plot_topomap` expects. Getting this wrong produces
a plausible-looking but scrambled topography, which is why the subset assertion is
here rather than left implicit.

In [ ]:
topo_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
info_fname = meta.loc[
    meta[pidx_col] == SUBJECT_INDICES[0], "SingleDataMetadata.FILENAME"
].iloc[0]
topo_info = (
    topo_handler.load_data_file(
        info_fname,
        is_processed=True,
        processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
        preload=False,
    )
    .pick("eeg")
    .info
)
name_pos = {n: i for i, n in enumerate(channel_names)}
assert set(topo_info["ch_names"]) <= set(name_pos), (
    "RAW_CROPPED channels are not a subset of the wavelet channel names."
)
info_order = [name_pos[n] for n in topo_info["ch_names"]]
print(f"Topomap montage: {len(topo_info['ch_names'])} electrodes "
      f"(reordered onto the wavelet channel order).")

## Step 6 — Per-IC, Per-Subject Stimulus-Locked TF Maps & Topomaps

One figure per participant, one column per IC: the **stimulus-locked TF map** on
top, its **scalp topography** below. The two rows answer different questions and
neither is sufficient alone — the TF map says *when and at which frequency* the
component is active, the topomap says *where on the scalp* it comes from. A
band-limited stripe over a single electrode or the rim is an artefact the ICA has
isolated, and the TF map alone would not distinguish it from a real response.

Markers on each TF map: dashed lines at **stimulus onset (t = 0)** and
**offset (t = 0.5 s)**, dotted green line at **`ASSR_FREQ`**. A 40 Hz steady state
should appear as a horizontal band on the green line, present between the dashed
lines and absent outside them.

**Colour scales.** Shared across a subject's TF maps, per-IC for the topomaps —
and that asymmetry is forced by the decomposition, not a style choice. FastICA's
`whiten="unit-variance"` fixes every source to unit variance, so the maps are
already on one scale and a shared limit is fair; the *amplitude* therefore lives
entirely in the pattern, so a shared topomap scale would render every low-variance
component as a blank disc. Compare topomap **shape** across panels and use the
`var` percentage in each title for weight. (Note this also means a peaky TF map and
a diffuse one can carry the same total energy — concentration is real information.)

This plots explicitly rather than calling the CLI script's
`plot_participant_components`: that renderer takes ASSR specificity scores and
annotates components that clear the threshold, which is exactly the analysis layer
that is not in this notebook.

In [ ]:
MAX_COMPONENT_COLS = 5   # ICs per row-pair before wrapping

# seaborn's whitegrid theme (set in the setup cell) sets axes.grid=True and
# axes.edgecolor='.8'. The first draws grid lines across every TF heatmap; the
# second washes out MNE's head outline, which it draws with
# rcParams["axes.edgecolor"]. Both figures are therefore drawn inside this
# context rather than under the notebook-wide theme.
_PLOT_RC = {"axes.grid": False, "axes.edgecolor": "black"}


def plot_subject_components(tf, pat, icvar, label, ret, plots_dir=None):
    """TF map over scalp topography for every IC of one subject.

    Args:
        tf: ``(K, F, win)`` component score maps, sign- and order-corrected.
        pat: ``(K, C)`` forward channel patterns, in wavelet-cache channel order.
        icvar: ``(K,)`` per-IC share of channel-space energy.
        label: participant label for the title and file name.
        ret: retained channel-variance fraction, for the title.
        plots_dir: directory to save into; None to only display.

    Returns:
        The Matplotlib figure.
    """
    n_k = tf.shape[0]
    ncols = min(MAX_COMPONENT_COLS, n_k)
    n_blocks = int(np.ceil(n_k / ncols))
    # Shared TF limit: fair because unit-variance sources share a scale.
    vmax = float(np.abs(tf).max()) or 1e-12
    extent = [epoch_times[0], epoch_times[-1], wavelet_freqs[0], wavelet_freqs[-1]]

    with plt.rc_context(_PLOT_RC):
        # Constrained layout: each TF panel carries axis labels directly above a
        # topomap with its own title, and a fixed grid collides the two.
        fig, axes = plt.subplots(
            2 * n_blocks, ncols, figsize=(3.4 * ncols, 6.4 * n_blocks),
            squeeze=False, layout="constrained",
        )
        tf_im = None
        for k in range(n_k):
            block, col = divmod(k, ncols)
            ax_tf, ax_topo = axes[2 * block, col], axes[2 * block + 1, col]

            tf_im = ax_tf.imshow(
                tf[k], aspect="auto", origin="lower", extent=extent,
                cmap="RdBu_r", vmin=-vmax, vmax=vmax,
            )
            ax_tf.axvline(0.0, color="k", ls="--", lw=0.8)              # onset
            ax_tf.axvline(AssrEpoch.STIMULUS_DURATION_S, color="k", ls="--", lw=0.8)
            ax_tf.axhline(ASSR_FREQ, color="lime", ls=":", lw=1.2)      # 40 Hz
            ax_tf.set_title(f"IC{k + 1}   var {icvar[k] * 100:.1f}%", fontsize=9)
            ax_tf.set_xlabel("Time rel. onset (s)", fontsize=8)
            ax_tf.set_ylabel("Frequency (Hz)", fontsize=8)

            # Per-IC symmetric scale: patterns carry the amplitude, so a shared
            # limit would flatten low-variance components into a blank disc.
            vlim = float(np.percentile(np.abs(pat[k]), 99)) or 1e-12
            mne.viz.plot_topomap(
                pat[k][info_order], topo_info, axes=ax_topo, show=False,
                cmap="RdBu_r", vlim=(-vlim, vlim), contours=4,
            )
            ax_topo.set_title(f"IC{k + 1} pattern (\u00b1{vlim:.2g})", fontsize=8)

        for k in range(n_k, n_blocks * ncols):      # blank the unused slots
            block, col = divmod(k, ncols)
            axes[2 * block, col].axis("off")
            axes[2 * block + 1, col].axis("off")

        fig.colorbar(tf_im, ax=axes.ravel().tolist(), shrink=0.4,
                     label="IC score (unit variance)")
        fig.suptitle(
            f"{label} \u2014 per-IC stimulus-locked TF map and scalp topography "
            f"({CONDITION.value}/{MUSIC_TYPE.value})\n"
            f"{n_k} ICs unmixed over {pat.shape[1]} channels; whitening "
            f"retained {ret * 100:.1f}% of channel variance; ICs ordered by "
            f"variance, "
            f"topomap scales are per-IC",
            fontsize=11,
        )
    if plots_dir is not None:
        fig.savefig(plots_dir / f"separate_ica_{label}_components.png", dpi=150)
    return fig


for k, label in enumerate(subject_labels):
    plot_subject_components(
        tf_maps[k], patterns[k], ic_variance[k], label, float(retained[k]),
        plots_dir=PLOTS_DIR,
    )
    plt.show()
print(f"Wrote {n_subj} figure(s) to {PLOTS_DIR}")

## Step 7 — Does an IC Track the Stimulus at 40 Hz?

For every IC, correlate its **40 Hz time course from the trial average** with a
binary stimulus regressor — `1` while the stimulus is on, `0` otherwise — over the
same epoch window the TF maps show. One number per IC: **|r|**, sorted descending.

The 40 Hz row is read straight out of `tf_maps`, so this step adds no data
loading; it is a summary of what the panels in Step 6 already show, reduced to one
comparable number per component.

**Why the absolute value.** Flipping an IC flips its time course and therefore the
sign of *r*, so signed *r* is not a property of the component — it is a property of
whichever convention set the sign (Step 4 uses the largest TF excursion, which has
nothing to do with 40 Hz). Taking |r| makes the measure **invariant to the ICA
sign ambiguity**, which is what lets the ranking mean something. The assertion in
the cell below pins that invariance down.

The flip side is that "40 Hz power rises while the stimulus is on" and "40 Hz power
falls while it is on" score identically — but under sign ambiguity those are the
same component seen two ways, so nothing real is lost. Which one a given IC is, is
readable off its TF map in Step 6.

**Read the magnitudes relatively, not absolutely.** The trial average is smooth by
construction, so correlating it against a box gives comfortably higher values than
single trials would, and even a perfect steady state cannot reach 1 — the box has
sharp edges the wavelet's temporal smoothing cannot follow. What matters is how
each IC compares with the others in the same panel.

In [ ]:
# Frequency row used for the correlation — the bin nearest ASSR_FREQ.
ASSR_BIN = int(np.argmin(np.abs(wavelet_freqs - ASSR_FREQ)))
box = stim_mask.astype(float)   # 1 while the stimulus is on, 0 otherwise


def stimulus_abs_correlation(tf, freq_bin, regressor):
    """|Pearson r| between each IC's time course at one frequency and a regressor.

    Args:
        tf: ``(..., n_ica, n_freqs, win)`` trial-averaged score maps.
        freq_bin: index of the frequency row to correlate.
        regressor: ``(win,)`` regressor, here the binary stimulus box.

    Returns:
        ``(..., n_ica)`` absolute correlations. NaN where a component's time
        course is constant. Absolute because flipping an IC flips the sign of r,
        so only the magnitude is a property of the component.
    """
    course = tf[..., freq_bin, :]                       # (..., n_ica, win)
    a = course - course.mean(axis=-1, keepdims=True)
    b = regressor - regressor.mean()
    denom = np.sqrt((a**2).sum(axis=-1) * (b**2).sum())
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(denom > 0.0, np.abs((a * b).sum(axis=-1) / denom), np.nan)


abs_r = stimulus_abs_correlation(tf_maps, ASSR_BIN, box)      # (S, n_ica)

# The measure must not depend on the arbitrary ICA sign — that is the whole reason
# for the absolute value. Negating every component must leave it unchanged.
np.testing.assert_allclose(
    abs_r, stimulus_abs_correlation(-tf_maps, ASSR_BIN, box), atol=1e-12,
    err_msg="|r| changed when the components were negated — the measure is not "
            "sign-invariant, so the ranking would depend on Step 4's convention.",
)

print(f"Frequency {wavelet_freqs[ASSR_BIN]:.0f} Hz (bin {ASSR_BIN}); regressor on "
      f"for {int(box.sum())}/{box.size} samples of the epoch\n")
for k, label in enumerate(subject_labels):
    rank = np.argsort(-abs_r[k])
    top = ", ".join(f"IC{i + 1} {abs_r[k][i]:.3f}" for i in rank[:3])
    print(f"  {label} (trial average of {n_used_w[SUBJECT_INDICES[k]]} stimuli): "
          f"{top}")

In [ ]:
fig, axes = plt.subplots(
    1, n_subj, figsize=(4.4 * n_subj, 0.42 * n_ica + 2.4), squeeze=False,
    sharex=True, layout="constrained",
)
for k, (ax, label) in enumerate(zip(axes[0], subject_labels)):
    rank = np.argsort(-abs_r[k])                   # descending |r|
    y = np.arange(n_ica)
    ax.barh(y, abs_r[k][rank], color="#4c72b0")
    ax.set_yticks(y, [f"IC{i + 1}" for i in rank], fontsize=8)
    ax.invert_yaxis()                              # largest at the top
    ax.set_xlim(0.0, max(1.05 * float(np.nanmax(abs_r)), 0.05))
    ax.set_xlabel("|Pearson r| with stimulus box")
    ax.set_title(
        f"{label}  (trial average, n={n_used_w[SUBJECT_INDICES[k]]} stimuli)\n"
        f"max |r| = {np.nanmax(abs_r[k]):.3f}",
        fontsize=10,
    )
    for yi, v in zip(y, abs_r[k][rank]):
        ax.text(v + 0.008, yi, f"{v:.3f}", va="center", fontsize=7)
axes[0, 0].set_ylabel("component (ordered by |r|)")
fig.suptitle(
    f"Trial-averaged {wavelet_freqs[ASSR_BIN]:.0f} Hz time course vs the stimulus "
    f"box \u2014 {CONDITION.value}/{MUSIC_TYPE.value}\n"
    f"absolute correlation, so the ranking does not depend on each IC's "
    f"arbitrary sign",
    fontsize=11,
)
fig.savefig(PLOTS_DIR / "separate_ica_stimulus_correlation.png", dpi=150)
plt.show()
print(f"Wrote {PLOTS_DIR / 'separate_ica_stimulus_correlation.png'}")